In this workflow we only cleanse and structure Tables from USGS 2019 Mineral Yearbook.

In [204]:
import pandas as pd
import os
import sys
sys.path.append("../../modules/structuring_tables")
from algorithm import get_consecutive_ranges
from getcolwidth import get_real_column_width
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string, get_column_letter 
import xlwings as xw
import re

In [205]:
# Create a list of file names referring to only tables 2019 MYB. 
continent_select = "asia"
input_dir = "../../data/original_table/usgs_" + continent_select + "/"
input_filenames_all = os.listdir(input_dir)
input_filenames_2019 = [name for name in input_filenames_all if "2019" in name]
# for name in input_filenames_2019:
    # Invoke functions

##### Take Single Country (2019) as an example (encapsulated into .py)

In [206]:
name = "China_2019.xlsx"

##### 1.1 Identify redundant rows to remove
*Important note: All rows must be remove at the same time, otherwise column index should be reloaded.*

(I) Penetrating merged cells

In [207]:
# name = "Bolivia_2019.xlsx"
filepath =  os.path.join(input_dir, name)
fullfile_openpyxl = load_workbook(filepath)
# Check if `fullfile_openpyxl.sheetnames` includes "Table 2"
# table_openpyxl = fullfile_openpyxl['Table 2']
table_openpyxl = list(fullfile_openpyxl)[2] # DEBUG: Table 2 are at times not named as 'Table 2' in Excel (e.g., in China-2019 as 'Table2')
merged_cells = list(table_openpyxl.merged_cells.ranges)

## Delete rows where merged cells penetrated through all columns
table_maxcol = merged_cells[0].bounds[2] # Determinator of penetrating merged cells
merged_cells_penetrate = [mc for mc in merged_cells if mc.bounds[2] == table_maxcol] # Select all penetrating merged cells
merged_cells_penetrate_rows = [mc_row.bounds[1] for mc_row in merged_cells_penetrate] # Locate rows of penetrating merged cells
merged_cells_penetrate_rows.sort(reverse=False)

print(merged_cells_penetrate_rows)

[1, 2, 3, 4, 5, 60, 61, 62, 63, 64, 65, 66, 67, 68, 123, 124, 125, 126, 127, 128, 129, 130, 131, 187, 188, 189, 190, 191, 192, 193, 194, 249, 250, 251, 252, 253, 254, 255, 256, 257, 313, 314, 315, 316, 317, 318, 319, 320, 375, 376, 377, 378, 379, 380, 381, 382, 383, 438, 439, 440, 441, 442, 443, 444, 445, 446, 502, 503, 504, 505, 506, 507, 508, 509, 565, 566, 567, 568, 569, 570, 571, 572, 629, 630, 631, 632, 633, 634, 635, 691, 692, 693, 694, 695, 696, 697, 698, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 818, 820, 821, 822, 823, 824, 879, 880, 881, 882, 883, 884, 885, 886, 887, 943, 944, 945, 946, 947, 948, 949, 950, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1056, 1057, 1058, 1059]


(II) Footnotes (Criteria: below the last lower border)

In [208]:
last_row_with_lowerborder = None
for row in range(table_openpyxl.max_row, 0, -1): # Iterate bottom-up
    for cell in table_openpyxl[row]:
        if cell.border.bottom.style: # Stop the loop as soon as the first cell with lower border is found.
            footnote_identify_regex = r"^(eEstimate|1[a-zA-Z])"
            next_cell_value = table_openpyxl[row+1][0].value
            if re.match(footnote_identify_regex, next_cell_value): # Determine if the next row is footnote
                last_row_with_lowerborder = row + 1 
            else:
                last_row_with_lowerborder = row + 2 # VERY IMPORTANT: cell.border.bottom.style can sometimes fail to identify frame layout of cells whose subsequent row contains merged cells.
    if last_row_with_lowerborder is not None:
        break
if last_row_with_lowerborder is not None:
    footnote_rows = list(range(last_row_with_lowerborder,table_openpyxl.max_row+1))
else:
    footnote_rows = []
    print("No footnote identified in this table! (Make sure to recheck the layout)")

print(row, last_row_with_lowerborder, table_openpyxl.max_row)
print(footnote_rows)

1055 1056 1223
[1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1105, 1106, 1107, 1108, 1109, 1110, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1118, 1119, 1120, 1121, 1122, 1123, 1124, 1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 1166, 1167, 1168, 1169, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1179, 1180, 1181, 1182, 1183, 1184, 1185, 1186, 1187, 1188, 1189, 1190, 1191, 1192, 1193, 1194, 1195, 1196, 1197, 1198, 1199, 1200, 1201, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1219, 

(III) Repeated headers

In [209]:
headers_rows = []
for row in table_openpyxl.iter_rows(min_col=1, max_col=1):
    cell = row[0]
    row_num = row[0].row
    if cell.value and "Commodit" in str(cell.value):
        if cell.border.top.style is not None:
            headers_rows.append(row_num)
        else:
            row_num_start = 1
            for r in range(row_num, 0, -1):
                previous_cell = table_openpyxl[r][0]
                if previous_cell.border.top.style:
                    row_num_start = r
                    break
            headers_rows = headers_rows + list(range(row_num_start, row_num+1))
print(headers_rows)

[6, 7, 69, 70, 132, 133, 195, 196, 258, 259, 321, 322, 384, 385, 447, 448, 510, 511, 573, 574, 636, 637, 699, 700, 762, 763, 825, 826, 888, 889, 951, 952, 1014, 1015]


(IV) Containing "—" or "Continued"

In [210]:
continue_rows = []
for row in table_openpyxl.iter_rows(min_col=1, max_col=1):
    cell = row[0]
    row_num = row[0].row
    if cell.value and ("—" in str(cell.value) or "continue" in str(cell.value).lower()):
        continue_rows.append(row_num)

print(continue_rows)

[64, 127, 134, 135, 190, 197, 198, 253, 260, 261, 316, 379, 386, 387, 442, 449, 450, 505, 512, 513, 568, 576, 631, 638, 694, 757, 764, 820, 883, 890, 891, 946, 1009, 1016, 1017]


(V) All upper letter (excluding "—" and "Continue(d)") in `Commodity` column

In [211]:
import re
allupper_rows = []
clean_regex = re.compile(r'—|continued?', re.IGNORECASE) # Construct regular expression to match keywords
for row in table_openpyxl.iter_rows(min_col=1, max_col=1):
    cell = row[0]
    row_num = row[0].row
    cell_value = row[0].value # Get value from the cell (return None the cell is empty)
    if cell_value and isinstance(cell_value, str): 
        cleaned_cell_value = clean_regex.sub('', cell_value).strip() # Remove all "—" and "continue(d)" if they ever exist
        if cleaned_cell_value and cleaned_cell_value.isupper():
            allupper_rows.append(row_num)

print(allupper_rows)

[1, 2, 64, 65, 127, 128, 190, 191, 253, 254, 316, 317, 379, 380, 442, 443, 505, 506, 568, 569, 631, 632, 694, 695, 757, 758, 820, 821, 883, 884, 946, 947, 1009, 1010]


**Gather all redundant rows and remove repeated ones:**

In [212]:
# Gather all redundant rows and remove repeated ones (with `list(set(list_A + list_B))`)
redundant_rows = list(set(merged_cells_penetrate_rows + footnote_rows + headers_rows + continue_rows + allupper_rows))
redundant_rows.sort(reverse=False) # Sort the rows in reverse order

# Concatenate list of located rows into a single string (consecutive series are simplified with colon like "1:5")
redundant_rows_concise = get_consecutive_ranges(redundant_rows) # Invoke function "get_consecutive_ranges(list)" from ../../module/structuring_tables/algorithm.py
redundant_rows_concise_str = ",".join(string for string in redundant_rows_concise)

print(redundant_rows_concise_str)

1:7,60:70,123:135,187:198,249:261,313:322,375:387,438:450,502:513,565:574,576:576,629:638,691:700,750:764,818:818,820:826,879:891,943:952,1006:1017,1056:1223


##### Step 1.2: Identify isolation columns to remove

(I) Apparent isolation

In [213]:
# Delete columns significantly narrower than its right neighbor (isolation strips)
# IMPORTANT: table_openpyxl.column_dimensions['D'].width can return false value 13.0 (default value at the back end of openpyxl when apparent width is not attainable, given width of adjacent columns are sometimes compressed in excel). Thus we encapsulated a module to decompress and reveal the real width.
real_col_width_list = get_real_column_width(table_openpyxl)
threshold_ratio = 0.3 ## NOTE: The threshold needs to be fine-tuned to tailor to layouts in different tables.
isolation_col = []
for col_idx in range(2, table_maxcol+1): # IMPORTANT: Start from the second column and end at the last column
    # Get reference letter (e.g., A/B/C...) of current column and its neighbors
    left_col_letter = get_column_letter(col_idx - 1)
    current_col_letter = get_column_letter(col_idx)
    right_col_letter = get_column_letter(col_idx + 1)

    # Get width of current column and its neighbors
    left_col_width = real_col_width_list[left_col_letter]
    current_col_width = real_col_width_list[current_col_letter]
    right_col_width = real_col_width_list[right_col_letter]

    if current_col_width < (right_col_width * threshold_ratio): # Compare solely with the right-hand column
        isolation_col.append(current_col_letter)

print(isolation_col)

['B', 'D', 'F', 'H']


(II) Isolation strip between `Commodity` column and `Unit` column

In [214]:
# Identify totally empty columns
empty_columns = []
for col in table_openpyxl.columns:
    is_empty = all(cell.value is None or str(cell.value).strip()=="" 
                   for cell in col)
    if is_empty:
        col_letter = get_column_letter(col[0].column)
        empty_columns.append(col_letter)
print(empty_columns)

['B', 'D', 'F', 'H']


In [215]:
redundant_cols = list(set(isolation_col + empty_columns))
redundant_cols.sort(reverse=True) # IMPORTANT: Columns must be removed from right to left

# Compile list of isolation strips into a single valid string comprehensible to xlwings package
redundant_col_str = ','.join([f"{col}:{col}" for col in redundant_cols])

print(redundant_col_str)

H:H,F:F,D:D,B:B


##### Step 1.3: Operate removal

In [216]:
# name = "Bolivia_2019.xlsx"
filepath =  os.path.join(input_dir, name)

# To save time, we initiate a new Excel process only if Excel is not running at the back end
if len(xw.apps) > 0:
    app = xw.apps.active
    print("Running Excel process is invoked :)")
else:
    app = xw.App(visible=True)
    print("New Excel process has started!")

# Open the original table
fullfile_xlwings = app.books.open(filepath) # Open .xls/xlsx file
# table_xlwings = fullfile_xlwings.sheets['Table 2'] # Refer to worksheet 'Table 2'
table_xlwings = fullfile_xlwings.sheets[2] # DEBUG: Table 2 are at times not named as 'Table 2' in Excel (e.g., in China-2019 as 'Table2')

## Delete all rows (in batch) where merged cells penetrate through all columns
table_xlwings.api.Range(redundant_rows_concise_str).Delete() # Highly optimized
## Delete all columns (in batch) serving as isolation strips
table_xlwings.api.Range(redundant_col_str).Delete() # [NEWLY ADDED]

# Save the modified copy and close the original doc
fullfile_xlwings.save("../../data/structured_tables/test/modified.xlsx")
fullfile_xlwings.close()

Running Excel process is invoked :)
